Version 1: Circuit that constructs the superposition: $\sum_{2^I}(|Window_i>|Index_i>|Grid>)$
- 1-dimensional network.
- Gray mask on idx (We disable gates: XX -> I). -> (New Improvement)
- Validity encoding on qubit 'p' (0=valid / 1=invalid). -> (New Improvement, Discontinued) 
- Filtering solutions with a conditional on P. -> (New Improvement, Discontinued)

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
print(qiskit.__version__)

In [ ]:
N = 4
M = 2
W = N - M + 1
k = int(np.ceil(np.log2(W))) if W > 1 else 1
print(f"Posibles soluciones W: {W}, qubits necesarios para representar W posiciones -> k: {k}")

In [ ]:
# VERSION OPTIMIZADA CON PENALIZACION (sin tocar el circuito original)
# Idea: mantenemos la optimizacion por mascara Gray en idx y, una vez cargada la
# ventana en m, penalizamos las ramas con m != 000 desviando amplitud a una ancilla.
# Al postseleccionar p = 0, las ventanas no nulas desaparecen por completo.

n_opt = QuantumRegister(N, "n")
idx_opt = QuantumRegister(k, "i")
m_opt = QuantumRegister(M, "m")
flag_opt = QuantumRegister(1, "f")
pen_opt = QuantumRegister(1, "p")
qc_opt = QuantumCircuit(n_opt, idx_opt, m_opt, flag_opt, pen_opt)

# ---------------------------
# Estado inicial aleatorio para n: entre 0 y N-M posiciones ocupadas, todas distintas
# rng_n = np.random.default_rng()
# num_occupied = int(rng_n.integers(0, N - M + 1))
# occupied_positions = sorted(rng_n.choice(N, size=num_occupied, replace=False).tolist()) if num_occupied > 0 else []

# for pos in occupied_positions:
#     qc_opt.x(n_opt[pos])

# print(f"Numero aleatorio de posiciones ocupadas en n: {num_occupied}")
# print(f"Posiciones ocupadas aleatorias en n: {occupied_positions}")
qc_opt.x(n_opt[1])
# ---------------------------

penalty_theta = np.pi / 2
target_zero_window = "0" * M

# Superposicion uniforme de indices
qc_opt.h(idx_opt)

# Orden Gray filtrado a ventanas validas 0..W-1
order = [g for t in range(1 << k) if (g := (t ^ (t >> 1))) < W]

def zero_mask(i, k):
    # 1 => ese idx[b] necesita X para representar control sobre 0
    return [1 if ((i >> b) & 1) == 0 else 0 for b in range(k)]

x_flips_opt = 0
current_mask = [0] * k

for i in order:
    target_mask = zero_mask(i, k)

    # Cambiar SOLO bits que difieren respecto a la mascara anterior
    for b in range(k):
        if target_mask[b] != current_mask[b]:
            qc_opt.x(idx_opt[b])
            x_flips_opt += 1

    current_mask = target_mask

    # XOR de la ventana seleccionada sobre m
    for j in range(M):
        controls = [idx_opt[b] for b in range(k)] + [n_opt[i + j]]
        qc_opt.mcx(controls, m_opt[j])

# Limpiar mascara al final para dejar idx como al principio del bloque
for b in range(k):
    if current_mask[b] == 1:
        qc_opt.x(idx_opt[b])
        x_flips_opt += 1

# Penalizacion coherente: marcar m != 000 y desviar parte de su amplitud a pen_opt
for q in m_opt:
    qc_opt.x(q)
qc_opt.mcx(list(m_opt), flag_opt[0])
qc_opt.x(flag_opt[0])
qc_opt.cry(2 * penalty_theta, flag_opt[0], pen_opt[0])


qc_opt.x(flag_opt[0])
qc_opt.mcx(list(m_opt), flag_opt[0])
for q in m_opt:
    qc_opt.x(q)

# Referencia de coste en X del enfoque original
x_flips_naive = 0
for i in range(W):
    z = sum(1 for b in range(k) if ((i >> b) & 1) == 0)
    x_flips_naive += 2 * z

print(f"Orden usado (Gray filtrado): {order}")
print(f"X en enfoque original: {x_flips_naive}")
print(f"X en enfoque optimizado: {x_flips_opt}")
print(f"Penalizacion sobre m != {target_zero_window}: theta = {penalty_theta:.4f} rad")
print(f"Factor de supervivencia al postseleccionar p=0: cos(theta)^2 = {np.cos(penalty_theta) ** 2:.6f}")

qc_opt.draw(output='mpl')

In [ ]:
sv_opt = Statevector(qc_opt)
sv_opt.draw(output='latex', max_size=2**qc_opt.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# Analisis del circuito optimizado con penalizacion.
# Como la ancilla p queda entrelazada con las ventanas no nulas, ya no esperamos
# que el estado reducido sobre (idx, m) sea puro. Por eso mostramos probabilidades
# totales y condicionadas a la rama no penalizada p = 0.

tol_n = 1e-10
total_q_opt = qc_opt.num_qubits
array_posiciones = None
prob_pen_0 = 0.0
prob_pen_1 = 0.0
aggregated = {}

for basis_idx, amp in enumerate(sv_opt.data):
    prob = float(np.abs(amp) ** 2)
    if prob <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q_opt}b')
    pen_bit = bits_full[0]
    flag_bit = bits_full[1]
    m_desc = bits_full[2:2 + M]
    i_desc = bits_full[2 + M:2 + M + k]
    n_desc = bits_full[-N:]

    ventana = m_desc[::-1]
    indices = i_desc
    n_asc = n_desc[::-1]

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("Los qubits n no son deterministas (no hay un unico array de posiciones).")

    if flag_bit != '0':
        raise ValueError("La ancilla flag no quedo descomputada al final del circuito.")

    key = (indices, ventana)
    if key not in aggregated:
        aggregated[key] = {'p0': 0.0, 'p1': 0.0}

    if pen_bit == '0':
        aggregated[key]['p0'] += prob
        prob_pen_0 += prob
    else:
        aggregated[key]['p1'] += prob
        prob_pen_1 += prob

if array_posiciones is None:
    raise ValueError("No se pudo inferir el array de posiciones desde el statevector.")

print(f"Array de posiciones = {array_posiciones}\n")
print(f"Probabilidad de p = 0: {prob_pen_0:.6f}")
print(f"Probabilidad de p = 1: {prob_pen_1:.6f}\n")

filas = []
for (indices, ventana), probs in aggregated.items():
    idx_int = int(indices, 2)
    p_total = probs['p0'] + probs['p1']
    p_cond = probs['p0'] / prob_pen_0 if prob_pen_0 > tol_n else 0.0
    filas.append((idx_int, indices, ventana, p_total, probs['p0'], probs['p1'], p_cond))

filas.sort(key=lambda x: (x[0], x[2]))

# Primeros W resultados: ventanas validas
for _, indices, ventana, p_total, p0, p1, p_cond in filas[:W]:
    print(
        f"Indices: {indices} || Ventana: {ventana} || "
        f"P_total={p_total:.6f} || P(p=0)={p0:.6f} || P(p=1)={p1:.6f} || P_cond(p=0)={p_cond:.6f}"
    )
    print()

# Resto de resultados (fuera del rango valido de ventanas)
if len(filas) > W:
    print("----------------------------------------")
    print("Resto (indices fuera de ventana valida):\n")
    for _, indices, ventana, p_total, p0, p1, p_cond in filas[W:]:
        print(
            f"Indices: {indices} || Ventana: {ventana} || "
            f"P_total={p_total:.6f} || P(p=0)={p0:.6f} || P(p=1)={p1:.6f} || P_cond(p=0)={p_cond:.6f}"
        )
        print()

In [ ]:
# Circuito dinamico: medir primero p y solo medir idx/m si p = 0.
# Esto implementa un intento de "repeat-until-success":
# - si p = 0, el estado ya colapso al subespacio valido y leemos idx
# - si p = 1, ese intento se descarta

shots = 4096

c_p_dyn = ClassicalRegister(1, 'c_p_dyn')
c_idx_dyn = ClassicalRegister(k, 'c_i_dyn')
c_m_dyn = ClassicalRegister(M, 'c_m_dyn')

qc_dynamic = qc_opt.copy()
qc_dynamic.add_register(c_p_dyn, c_idx_dyn, c_m_dyn)
qc_dynamic.measure(pen_opt, c_p_dyn)

with qc_dynamic.if_test((c_p_dyn[0], 0)):
    qc_dynamic.measure(idx_opt, c_idx_dyn)
    qc_dynamic.measure(m_opt, c_m_dyn)

backend = AerSimulator()
result_dyn = backend.run(qc_dynamic, shots=shots, memory=True).result()
counts_dyn = result_dyn.get_counts()
memory_dyn = result_dyn.get_memory()

accepted_counts = {}
accepted_shots = 0
rejected_shots = 0

for bitstring in memory_dyn:
    pen_bits, m_bits, idx_bits = bitstring.split()
    if pen_bits == '0':
        accepted_shots += 1
        window_bits = m_bits[::-1]
        key = (idx_bits, window_bits)
        accepted_counts[key] = accepted_counts.get(key, 0) + 1
    else:
        rejected_shots += 1

accepted_sorted = sorted(accepted_counts.items(), key=lambda x: x[1], reverse=True)

print(f"Shots totales: {shots}")
print(f"Intentos validos (p=0): {accepted_shots}")
print(f"Intentos descartados (p=1): {rejected_shots}")
print(f"Tasa empirica de exito por intento: {accepted_shots / shots:.6f}")
print(f"Tasa teorica de exito por intento: {prob_pen_0:.6f}")
print("\nIndices obtenidos solo en intentos validos:")

for (idx_bits, window_bits), count in accepted_sorted[:10]:
    print(
        f"idx={idx_bits} || ventana={window_bits} || "
        f"counts={count} || prob_cond~={count / accepted_shots:.6f}"
    )

qc_dynamic.draw(output='mpl')

In [ ]:
# Simulacion clasica del esquema repeat-until-success.
# Usamos directamente la distribucion teorica ya calculada:
# - cada intento tiene probabilidad prob_pen_0 de dar p = 0
# - condicionado a p = 0, idx sigue la distribucion p_cond de las filas validas

num_experiments = 1000
max_attempts = 10
rng = np.random.default_rng(12345)

valid_rows = [(idx_bits, window_bits, p_cond) for _, idx_bits, window_bits, _, _, _, p_cond in filas if p_cond > 0]
valid_labels = [(idx_bits, window_bits) for idx_bits, window_bits, _ in valid_rows]
valid_probs = np.array([p_cond for _, _, p_cond in valid_rows], dtype=float)
valid_probs = valid_probs / valid_probs.sum()

successes = 0
attempts_to_success = []
valid_idx_counts = {}

for _ in range(num_experiments):
    for attempt in range(1, max_attempts + 1):
        if rng.random() < prob_pen_0:
            successes += 1
            attempts_to_success.append(attempt)
            chosen = rng.choice(len(valid_labels), p=valid_probs)
            idx_bits, window_bits = valid_labels[chosen]
            key = (idx_bits, window_bits)
            valid_idx_counts[key] = valid_idx_counts.get(key, 0) + 1
            break

success_probability_emp = successes / num_experiments
success_probability_theory = 1 - (1 - prob_pen_0) ** max_attempts
avg_attempts_emp = np.mean(attempts_to_success) if attempts_to_success else float('nan')
avg_attempts_theory = sum(attempt * prob_pen_0 * (1 - prob_pen_0) ** (attempt - 1) for attempt in range(1, max_attempts + 1))
valid_idx_sorted = sorted(valid_idx_counts.items(), key=lambda x: x[1], reverse=True)

print(f"Experimentos independientes: {num_experiments}")
print(f"Maximo de intentos por experimento: {max_attempts}")
print(f"Probabilidad empirica de encontrar un indice valido: {success_probability_emp:.6f}")
print(f"Probabilidad teorica de encontrar un indice valido: {success_probability_theory:.6f}")
print(f"Numero medio de intentos hasta exito: {avg_attempts_emp:.6f}")
print(f"Media teorica truncada de intentos: {avg_attempts_theory:.6f}")
print("\nFrecuencias de indices validos encontrados:")

for (idx_bits, window_bits), count in valid_idx_sorted:
    print(
        f"idx={idx_bits} || ventana={window_bits} || "
        f"counts={count} || prob~={count / successes:.6f}"
    )

In [ ]:
# Trazas detalladas de varios experimentos repeat-until-success.
# Cada experimento ejecuta intentos hasta encontrar una solucion valida.
# El output muestra la secuencia completa de resultados hasta el primer exito.

num_trace_experiments = 12
rng_trace = np.random.default_rng(2026)

print(f"Probabilidad de exito por intento: {prob_pen_0:.6f}\n")

for exp_id in range(1, num_trace_experiments + 1):
    print(f"Experimento {exp_id}")
    attempt = 0

    while True:
        attempt += 1

        if rng_trace.random() < prob_pen_0:
            chosen = rng_trace.choice(len(valid_labels), p=valid_probs)
            idx_bits, window_bits = valid_labels[chosen]
            print(
                f"  Intento {attempt}: p=0 -> solucion valida encontrada, "
                f"idx={idx_bits}, ventana={window_bits}"
            )
            print(f"  Total de intentos hasta exito: {attempt}\n")
            break

        print(f"  Intento {attempt}: p=1 -> descartar y repetir")